# Flower Classification - Feature Engineering
## Creating and Transforming Features

This notebook creates additional features to improve model performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Load Original Data

In [ ]:
# Load original dataset (before scaling)
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target

print(f'Data shape: {df.shape}')
df.head()

## 2. Create Ratio Features

In [ ]:
# Create ratio features between length and width
df['sepal_length_to_width_ratio'] = df['sepal length (cm)'] / (df['sepal width (cm)'] + 1e-6)
df['petal_length_to_width_ratio'] = df['petal length (cm)'] / (df['petal width (cm)'] + 1e-6)

print('Created ratio features:')
print('  - sepal_length_to_width_ratio')
print('  - petal_length_to_width_ratio')
df[['sepal_length_to_width_ratio', 'petal_length_to_width_ratio']].describe()

## 3. Create Size Features

In [ ]:
# Sepal area and perimeter (approximation)
df['sepal_area'] = df['sepal length (cm)'] * df['sepal width (cm)']
df['sepal_perimeter'] = 2 * (df['sepal length (cm)'] + df['sepal width (cm)'])

# Petal area and perimeter (approximation)
df['petal_area'] = df['petal length (cm)'] * df['petal width (cm)']
df['petal_perimeter'] = 2 * (df['petal length (cm)'] + df['petal width (cm)'])

print('Created size features:')
print('  - sepal_area, sepal_perimeter')
print('  - petal_area, petal_perimeter')
df[['sepal_area', 'sepal_perimeter', 'petal_area', 'petal_perimeter']].describe()

## 4. Create Interaction Features

In [ ]:
# Create interaction features between sepal and petal measurements
original_features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

for i, col1 in enumerate(original_features):
    for col2 in original_features[i+1:]:
        new_col = f'{col1.split()[0]}_x_{col2.split()[0]}'
        df[new_col] = df[col1] * df[col2]

print('Created interaction features:')
interaction_cols = [col for col in df.columns if '_x_' in col]
for col in interaction_cols:
    print(f'  - {col}')

df[interaction_cols].describe()

## 5. Create Polynomial Features

In [ ]:
# Create squared features
for feature in original_features:
    squared_name = f'{feature.split()[0]}_squared'
    df[squared_name] = df[feature] ** 2

print('Created polynomial features:')
poly_cols = [col for col in df.columns if 'squared' in col]
for col in poly_cols:
    print(f'  - {col}')

df[poly_cols].describe()

## 6. Feature Summary Statistics

In [ ]:
# Get summary statistics for all numeric features
numeric_df = df.select_dtypes(include=[np.number])
summary = numeric_df.describe().T
summary['skewness'] = numeric_df.skew()
summary['kurtosis'] = numeric_df.kurtosis()

print(f'Total features after engineering: {len(numeric_df.columns)}')
summary[['mean', 'std', 'skewness', 'kurtosis']].sort_values('skewness', key=abs, ascending=False).head(10)

## 7. Visualize New Features

In [ ]:
# Plot distribution of ratio features by species
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='sepal_length_to_width_ratio', hue='species', alpha=0.6, kde=True, ax=axes[0])
axes[0].set_title('Sepal Length/Width Ratio')

sns.histplot(data=df, x='petal_length_to_width_ratio', hue='species', alpha=0.6, kde=True, ax=axes[1])
axes[1].set_title('Petal Length/Width Ratio')

plt.tight_layout()
plt.show()

In [ ]:
# Plot area features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='species', y='sepal_area', ax=axes[0])
axes[0].set_title('Sepal Area by Species')

sns.boxplot(data=df, x='species', y='petal_area', ax=axes[1])
axes[1].set_title('Petal Area by Species')

plt.tight_layout()
plt.show()

## 8. Correlation Analysis with New Features

In [ ]:
# Calculate correlation with target
df_with_target = df.copy()
correlations = df_with_target.corr()['species'].drop('species').abs().sort_values(ascending=False)

print('Top 15 features by correlation with target:')
print(correlations.head(15))

In [ ]:
# Heatmap of top correlated features
top_features = correlations.head(10).index.tolist()
top_features.append('species')

plt.figure(figsize=(12, 10))
corr_matrix = df[top_features].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', center=0, square=True)
plt.title('Correlation Heatmap - Top Features')
plt.tight_layout()
plt.show()

## 9. Save Engineered Features

In [ ]:
# Save engineered features dataframe
from pathlib import Path

data_dir = Path('../data/processed')
data_dir.mkdir(parents=True, exist_ok=True)

# Save full dataframe with engineered features
df.to_csv(data_dir / 'data_with_engineered_features.csv', index=False)

# Save list of engineered feature columns
engineered_columns = [col for col in df.columns if col not in ['sepal length (cm)', 'sepal width (cm)', 
                                                                 'petal length (cm)', 'petal width (cm)', 'species']]
import joblib
joblib.dump(engineered_columns, data_dir / 'engineered_columns.joblib')

print(f'Saved {len(df.columns)} total columns')
print(f'Added {len(engineered_columns)} engineered features')
print('Files saved:')
print('  - data_with_engineered_features.csv')
print('  - engineered_columns.joblib')

## Summary

✅ Created **ratio features**: sepal and petal length-to-width ratios  
✅ Created **size features**: area and perimeter approximations  
✅ Created **interaction features**: products between original features  
✅ Created **polynomial features**: squared terms  
✅ Total features: 4 original → 20+ engineered features  
✅ Analyzed correlations and saved engineered dataset